In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

# Proyecto: 

## Justificación del problema

La empresa M&S cuenta con un aplicativo web para la gestión de productos químicos. En la base de datos, se cuenta con información relacionada a los productos directamente relacionada con la información de las Fichas de Datos de Seguridad (FDS), que hace la función de la hoja de vida del producto. Debido a la gran variabilidad en la forma en que estos pueden ser nombrados, se tiene un problema actual en donde se cuenta con aproximadamente el 7% de productos repetidos y tiempos altos en la consolidación de inventarios. 

Los clientes que usan el aplicativo, cuentan con información relacionada al nombre interno que la organización destina para el producto. De esta manera, un mismo producto puede aparecer registrado con múltiples variantes de nombre, diferencias en abreviaturas, idioma, formato o referencia comercial. Por ejemplo, un mismo producto puede registrarse como:

* NaOH 0.1N Cleaning Solution

* 0.1 N Sodium Hydroxide Solution

* NaOH 0.1N cuvette cleaning solution

Ante este contexto, es de alta importancia desarrollar mecanismos que permitan identificar automáticamente qué productos del catálogo existente son más similares a un nuevo registro, a partir de la información disponible como el nombre del producto y su fabricante.

Un sistema que sugiera coincidencias probables permitiría:

* reducir la creación de duplicados,

* mejorar la calidad y consistencia del catálogo de productos,

* facilitar la búsqueda y recuperación de información dentro del sistema,

* optimizar procesos de gestión de inventarios químicos.

Por esta razón, resulta relevante explorar el uso de técnicas de aprendizaje automático y representaciones semánticas del texto que permitan calcular similitudes entre productos y sugerir automáticamente las coincidencias más probables dentro del catálogo existente.

## Objetivo
Desarrollar un modelo de Deep Learning, el cual con el nombre y fabricante de un producto químico ingresado por el usuario, identifique los productos más similares en el catálogo existente, devolviendo un ranking de coincidencias probables.

## Metodología

1. Estructura inicial de los datos

    El dataset de productos se divide en dos subconjuntos dependiendo de la disponibilidad de información adicional.

    * df_con_sin_o_uso: Productos que tienen información en uso o sinónimo. Tamaño aproximado: (50621, 5)

    * df_sin_sin_o_uso: Productos que NO tienen información ni en sinonimos ni en uso. Tamaño aproximado: (13874, 5)

    La motivación de la separación nace porque los productos que tienen más información contextual permiten construir mejores representaciones semánticas para el modelo.

    Sin embargo, ambos datasets siguen siendo parte del mismo catálogo y pueden utilizarse en el sistema final.

2. Limpieza y normalización de texto

    Se aplican transformaciones para estandarizar el texto:

    - convertir texto a minúsculas
    - eliminar caracteres especiales innecesarios
    - eliminar espacios duplicados
    - reemplazar valores nulos por cadenas vacías

3. Construcción del texto base del producto

    - Para cada producto se construye un texto consolidado con la información disponible.

        - texto_producto = nombre_producto + fabricante + sinonimos + uso

    - Si un producto no tiene sinonimos o uso, se construye únicamente con la información disponible.

4. Vectorización de productos
   - El texto consolidado de cada producto se transforma en una representación numérica.
   - Esta vectorización permite comparar productos en un espacio común.
   - En esta etapa todavía no se hace la predicción final, sino que se prepara la información para agrupar y analizar.

5. Clustering del catálogo
   - Sobre los vectores generados se aplica un algoritmo de clustering.
   - El objetivo es identificar grupos naturales de productos similares dentro del catálogo.

    Propósitos del clustering:
    - encontrar agrupaciones de productos parecidos
    - identificar posibles familias de productos
    - reducir el espacio de búsqueda para la predicción posterior
    - detectar registros potencialmente duplicados o muy cercanos

6. Análisis de clusters
   - Se revisa la composición de cada cluster.
   - Se analiza si los productos agrupados comparten patrones de nombre, fabricante, sinónimos o uso.
   - Se valida si los clusters realmente separan bien tipos de productos.
   - A partir de este análisis se pueden ajustar variables, limpieza o estrategia de vectorización.

7. Modelo de predicción / recuperación
   - Una vez estructurado el catálogo mediante clusters, se construye el sistema de predicción.
   - Cuando entra un producto nuevo:
       1. se limpia el texto de entrada
       2. se construye su texto consolidado
       3. se vectoriza usando la misma lógica del catálogo
       4. se identifica el cluster más probable o más cercano
       5. se compara solo contra los productos de ese cluster, o contra todo el catálogo si se requiere
       6. se genera un ranking de similitud


8. Evaluación del sistema
   - Se revisa si las sugerencias son coherentes.
   - Se valida si el producto correcto aparece dentro del top 5, top 10 o top 20.
   - Se analiza el aporte del clustering frente a una búsqueda directa sin agrupación previa.

9. Exposición mediante API
   - Se implementa un servicio con FastAPI.
   - El servicio recibe:
       nombre_producto
       fabricante
   - Internamente:
       limpia
       vectoriza
       identifica cluster
       calcula similitud
       devuelve ranking

   Salida:
       lista de productos más similares con score

## Carga de datos

In [2]:
df = pd.read_excel("listado productos.xlsx").iloc[:, 1:]
df

,id,nombre_producto,sinonimos,uso,fabricante
0,30818,0 1 m dtt,superscript iv first strand synthesis system |...,para uso en investigacion unicamente,thermofisher scientific
1,56997,0 1n naoh compartimiento r1b,access hstni b52699,NaN,beckman coulter
2,10421,0 1n naoh cuvette cleaning solution b,7098 10445578,agentes de diagn oacute stico,siemens healthcare
3,42583,0 45 m 47mm white gridded s pak filter membran...,NaN,investigacion y analisis bioquimicos,merck
4,56156,0 5 medios de crecimiento b de postgate modifi...,NaN,NaN,grupo volante
...,...,...,...,...,...
64490,35967,zytiga,zytiga abiraterone acetate abiraterone acetate...,producto farmaceutico terminado grupo farmacot...,janssen
64491,1360,zytokil 10mg 5ml,doxorubicina,no registra,pisa farmaceutica
64492,26325,zz76 01009 citrus passion,NaN,NaN,ungerer & company
64493,62579,zzliteprop prime 108 alt code l426802 00,NaN,NaN,grupo transmerquim


In [3]:
# Separar el DataFrame en dos
# Uno con al menos un sinónimo o un uso
df_con_sin_o_uso = df[(df['sinonimos'].notna() & (df['sinonimos'] != '')) | (df['uso'].notna() & (df['uso'] != ''))]

# El otro con los demás
df_sin_sin_o_uso = df[~((df['sinonimos'].notna() & (df['sinonimos'] != '')) | (df['uso'].notna() & (df['uso'] != '')))]

print("DataFrame con sinónimo o uso:")
print(df_con_sin_o_uso.shape)
print("\nDataFrame sin sinónimo o uso:")
print(df_sin_sin_o_uso.shape)

DataFrame con sinónimo o uso:
(50621, 5)

DataFrame sin sinónimo o uso:
(13874, 5)


## 3. Construcción del texto base 

In [4]:
# Seleccionar columnas categóricas para clustering
# Completamos los valores faltantes con cadena vacía para evitar problemas con los modelos venideros

df_modelo = df.copy()

df_modelo["nombre_producto"] = df_modelo["nombre_producto"].fillna("")
df_modelo["sinonimos"] = df_modelo["sinonimos"].fillna("")
df_modelo["uso"] = df_modelo["uso"].fillna("")
df_modelo["fabricante"] = df_modelo["fabricante"].fillna("")

In [5]:
# Crear una nueva columna que combine las columnas de texto para el clustering

df_modelo["texto_producto"] = (
    df_modelo["nombre_producto"] + " " +
    df_modelo["nombre_producto"] + " " +   # duplicamos nombre para darle peso
    df_modelo["fabricante"] + " " +
    df_modelo["sinonimos"] + " " +
    df_modelo["uso"]
).str.lower().str.replace(r"\s+", " ", regex=True).str.strip()

df_modelo[["id", "texto_producto"]].head(10)

,id,texto_producto
0,30818,0 1 m dtt 0 1 m dtt thermofisher scientific su...
1,56997,0 1n naoh compartimiento r1b 0 1n naoh compart...
2,10421,0 1n naoh cuvette cleaning solution b 0 1n nao...
3,42583,0 45 m 47mm white gridded s pak filter membran...
4,56156,0 5 medios de crecimiento b de postgate modifi...
5,36322,0 ftu turbidity calibration standard 0 ftu tur...
6,66080,000000026354 hth ph minus 000000026354 hth ph ...
7,43858,000000026354 hth ph minus 000000026354 hth ph ...
8,11166,000113 b iris 000113 b iris firmenich
9,11160,000136 btr bergamote 000136 btr bergamote firm...


## 4. Vectorización de texto

### Usando TF-IDF

In [6]:
# Vectorizar el texto utilizando TF-IDF
vectorizador_tfidf = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=30000
)

X_tfidf = vectorizador_tfidf.fit_transform(df_modelo["texto_producto"])

print(type(X_tfidf))
print(X_tfidf.shape)

<class 'scipy.sparse._csr.csr_matrix'>
(64495, 30000)


### Usando embeddings

In [7]:
# pip install sentence-transformers

In [ ]:
# Importar el modelo de Sentence Transformers
# El modelo permite trabajo de embbedings de texto, lo que es útil para clustering semántico en ingles y español. 
# Se ajusta a los datos que se están trabajando.
from sentence_transformers import SentenceTransformer

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Cargar el modelo preentrenado
modelo_embeddings = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 23823.79it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# Generar los embeddings para el texto del producto
embeddings = modelo_embeddings.encode(
    df_modelo["texto_producto"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)

print(type(embeddings))
print(embeddings.shape)

Batches: 100%|██████████| 1008/1008 [01:31<00:00, 10.99it/s]

<class 'numpy.ndarray'>
(64495, 384)


## 5. Clustering de los datos